# REDACT — Jailbreak Augmentation

**Stage 3 of 3.** Augments input prompts with jailbreak technique combinations and merges
everything into the complete dataset.

Technique families:
- **Obfuscation** — encoding, translation, structural wrapping, ascii art, tokenbreak, suffixes
- **Hacking** — persona roleplay, hypothetical framing, cognitive techniques
- **Manipulation** — FSH (few-shot hacking), DAP (distract and persuade)
- **Requests** — output-format / continuation / indirect / distractor structural attacks

`generate_jailbreaks()` runs in two phases: a deterministic **plan** (every sample ×
iteration gets a combination, written to a resumable JSONL manifest) and a batched
**execute** (LLM-dependent steps are pooled per model per round). Output is appended per
chunk, so a crash resumes from the output CSV.

**Input source** is configurable: either the constitution-seeded prompts
(`constitution_generation.ipynb`) or the content-moderation prompts
(`content_moderation.ipynb`).

**Handoff artifacts:** `Datasets/jailbreaks.csv` (per-unit jailbreaks) and
`Datasets/complete_dataset.csv` (inputs + outputs + jailbreaks merged).

Requires `VENICE_API_KEY` when `PURE_ONLY=False` (LLM-dependent techniques).

In [ ]:
from redact import set_seed
set_seed(42)

## Configuration

Adjust these settings before running. Small values are set for demo purposes.

In [ ]:
# --- Model ---
MODEL = "venice-uncensored-vllm"   # Generation model for LLM-dependent techniques

# --- Input source ---
INPUT_SOURCE = "constitution"      # "constitution" | "content_moderation"
MAX_SAMPLES = 20                   # Optional head cap on source prompts (None = all)

# --- Jailbreak sampler ---
MAX_COMPLEXITY  = 6                # Max total complexity per combination
MAX_OBFUSCATIONS = 2              # Max obfuscation families per combination
PURE_ONLY       = True            # True = no LLM calls (fast); False = full pool
ENTRY_TYPES     = ["harmful", "dual_use_harmful"]  # Filter source by entry_type (None = all)
ITERATIONS      = 1               # Combinations assigned per sample (1 = single-round)
CHUNK_SIZE      = 256             # Manifest units per engine batch
JB_SEED         = 42              # Combined with each sample id + iteration

## Step 1: Load Source Input Prompts

Loads the merged handoff dataset from the chosen upstream stage. Run that stage's notebook
first so the CSV exists.

In [ ]:
import pandas as pd
from redact import get_output_dir
from redact.dataset import merge_all

if INPUT_SOURCE == "constitution":
    src_path = get_output_dir() / "Datasets" / "constitution_inputs_merged.csv"
    if not src_path.exists():
        raise FileNotFoundError(
            f"{src_path} not found — run constitution_generation.ipynb first."
        )
    source = pd.read_csv(src_path)
    print(f"Loaded {len(source)} constitution-seeded prompts from {src_path.name}")
elif INPUT_SOURCE == "content_moderation":
    source = merge_all(accepted_only=True)
    if source.empty:
        raise ValueError("No content-moderation inputs found — run content_moderation.ipynb first.")
    print(f"Loaded {len(source)} content-moderation prompts via merge_all()")
else:
    raise ValueError(f"Unknown INPUT_SOURCE: {INPUT_SOURCE!r}")

if MAX_SAMPLES:
    source = source.head(MAX_SAMPLES)

print(f"\nUsing {len(source)} source prompts")
if "entry_type" in source.columns:
    print("=== By Entry Type ===")
    print(source["entry_type"].value_counts().to_string())
source.head(5)

## Step 2: Generate Jailbreaks

Each prompt gets a randomly sampled valid technique combination (compatibility rules from
`combination_spec.json` are enforced by the sampler). With `PURE_ONLY=True` no API calls
are made; set it to `False` to include translation, typos, cognitive hacking, manipulation,
etc. Saved per chunk to `Datasets/jailbreaks.csv`.

In [ ]:
from redact import generate_jailbreaks

jailbreaks = generate_jailbreaks(
    inputs=source,
    max_complexity=MAX_COMPLEXITY,
    max_obfuscations=MAX_OBFUSCATIONS,
    seed=JB_SEED,
    pure_only=PURE_ONLY,
    entry_types=ENTRY_TYPES,
    iterations=ITERATIONS,
    chunk_size=CHUNK_SIZE,
    model=MODEL,
)

print(f"\nGenerated {len(jailbreaks)} jailbreak rows")
if not jailbreaks.empty:
    print("\n=== Top techniques used ===")
    print(jailbreaks["technique"].value_counts().head(10).to_string())
    if "complexity" in jailbreaks.columns:
        print("\n=== Complexity distribution ===")
        print(jailbreaks["complexity"].value_counts().sort_index().to_string())
    if "is_noop" in jailbreaks.columns:
        print(f"\nNo-ops: {int(jailbreaks['is_noop'].sum())} / {len(jailbreaks)}")
    if "accepted" in jailbreaks.columns:
        print(f"Accepted: {int(jailbreaks['accepted'].sum())} / {len(jailbreaks)}")
jailbreaks.head(10)

## Step 3: Build Complete Dataset

Merges all generated data (content-moderation inputs, output responses, jailbreaks) from
their saved CSVs into a single dataset at `Datasets/complete_dataset.csv`.

In [ ]:
from redact import build_dataset

dataset = build_dataset()

print(f"\nComplete dataset: {len(dataset)} samples")
if not dataset.empty:
    if "dataset_type" in dataset.columns:
        print("\n=== By Dataset Type ===")
        print(dataset["dataset_type"].value_counts().to_string())
    if "category" in dataset.columns:
        print("\n=== By Category ===")
        print(dataset["category"].value_counts().to_string())
    if "technique" in dataset.columns:
        jb = dataset[dataset["dataset_type"] == "jailbreak"] if "dataset_type" in dataset.columns else dataset
        print("\n=== By Technique (jailbreaks only) ===")
        print(jb["technique"].value_counts().head(15).to_string())
dataset.head(20)